# Create Products Embedding for Customer Care

The objective of this notebook is to create embeddings of our products' descriptions so the AI assistant can find articles more relevant to user query. 

It will also help us fight the cold start problem in case user is a new customer or doesn't have a membership.

You can follow this [guide](https://ollama.com/blog/ollama-is-now-available-as-an-official-docker-image) to create and use Ollama models.

You can run: 
```
docker run -d --gpus=all --network stylistai-net -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama
```

Then, to create your qdrant vectorbase run the following command:
```
docker run -d --network stylistai-net --name qdrant --restart always \
-p 6333:6333 -p 6334:6334 \
-v your/desired/path/to/qdrant/storage:/qdrant/storage \
qdrant/qdrant
```

## Load Products Dataset

In [11]:
import pandas as pd

df_path = '/data/articles.csv'#'/data/processed/articles_df.csv'

df = pd.read_csv(df_path)
df.head(3)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [12]:
df.shape

(105542, 25)

In [13]:
df.columns

Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')

In [16]:
df.loc[0]

article_id                                                    108775015
product_code                                                     108775
prod_name                                                     Strap top
product_type_no                                                     253
product_type_name                                              Vest top
product_group_name                                   Garment Upper body
graphical_appearance_no                                         1010016
graphical_appearance_name                                         Solid
colour_group_code                                                     9
colour_group_name                                                 Black
perceived_colour_value_id                                             4
perceived_colour_value_name                                        Dark
perceived_colour_master_id                                            5
perceived_colour_master_name                                    

In [21]:
df['section_name'].nunique()

56

In [22]:
df['section_name'].unique()

array(['Womens Everyday Basics', 'Womens Lingerie',
       'Womens Nightwear, Socks & Tigh', 'Baby Essentials & Complements',
       'Men Underwear', 'Mama', 'Womens Small accessories',
       'Men H&M Sport', 'Kids Boy', 'Divided Basics',
       'Girls Underwear & Basics', 'Mens Outerwear',
       'Womens Big accessories', 'Divided Accessories',
       'Womens Swimwear, beachwear', 'Divided Selected',
       'Boys Underwear & Basics', 'Contemporary Street',
       'Contemporary Casual', 'Men Accessories', 'Men Suits & Tailoring',
       'Womens Everyday Collection', 'Men Shoes', 'Young Boy', 'H&M+',
       'Divided Collection', 'Ladies Denim', 'Contemporary Smart',
       'Womens Trend', 'Kids Outerwear', 'Young Girl', 'Womens Shoes',
       'Womens Tailoring', 'Divided Projects', 'Denim Men', 'Men Other',
       'Womens Jackets', 'Men Other 2', 'Baby Boy', 'Womens Casual',
       'Kids Accessories, Swimwear & D', 'Ladies H&M Sport',
       'Kids & Baby Shoes', 'Baby Girl', 'Kids Girl

In [24]:
df['index_name'].nunique()

10

In [25]:
df['index_name'].unique()

array(['Ladieswear', 'Lingeries/Tights', 'Baby Sizes 50-98', 'Menswear',
       'Ladies Accessories', 'Sport', 'Children Sizes 92-140', 'Divided',
       'Children Sizes 134-170', 'Children Accessories, Swimwear'],
      dtype=object)

## Create Embeddings

### Qdrant set up

In [31]:
from qdrant_client import QdrantClient
QDRANT_URL = "http://qdrant:6333"

client = QdrantClient(url=QDRANT_URL)

In [10]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name="fashion_articles",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )

True

### Call Ollama Model

In [ ]:
import asyncio
import httpx
import uuid
from tqdm.asyncio import tqdm
from qdrant_client import AsyncQdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

# --- Configuration ---
OLLAMA_URL = "http://ollama:11434/api/embeddings"
CONCURRENT_REQUESTS = 10
BATCH_SIZE = 100

# --- Async Function to Get Embedding ---
async def get_embedding(client, semaphore, row):
    async with semaphore:
        prompt = f"""Product Name: {row['prod_name']}
        Product Type: {row['product_type_name']}
        Product Group: {row['product_group_name']}
        Color: {row['colour_group_name']}
        Description: {row['detail_desc']}"""

        payload = {
            "model": "embeddinggemma",
            "prompt": prompt
        }
        
        try:
            response = await client.post(OLLAMA_URL, json=payload, timeout=30.0)
            response.raise_for_status()
            embedding = response.json()['embedding']

            return PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding,
                payload={
                    "article_id": str(row['article_id']),
                    "prod_name": row['prod_name'],
                    "description": row['detail_desc'],
                    "article_index": row['index_name'],
                    "article_section": row['section_name'],
                }
            )
        except Exception as e:
            print(f"Error processing {row['article_id']}: {e}")
            return None


In [36]:
# --- Main Async Loop ---

async def run_pipeline(df):
    async_qdrant = AsyncQdrantClient(url=QDRANT_URL)

    if not await async_qdrant.collection_exists("fashion_articles"):
        await async_qdrant.create_collection(
            collection_name="fashion_articles",
            vectors_config=VectorParams(size=768, distance=Distance.COSINE)
        )

    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
    total_rows = len(df)

    pbar = tqdm(total=total_rows, desc="Processing Articles")

    async with httpx.AsyncClient() as http_client:
        tasks = []

        for _, row in df.iterrows():
            tasks.append(get_embedding(http_client, semaphore, row))

            if len(tasks) >= BATCH_SIZE:
                results = await asyncio.gather(*tasks)
                valid_points = [r for r in results if r is not None]
                
                if valid_points:
                    await async_qdrant.upsert(
                        collection_name="fashion_articles",
                        points=valid_points
                    )

                pbar.update(len(tasks))
                tasks = []

        if tasks:
            results = await asyncio.gather(*tasks)
            valid_points = [r for r in results if r is not None]
            if valid_points:
                await async_qdrant.upsert(collection_name="fashion_articles", points=valid_points)
            pbar.update(len(tasks))

    pbar.close()
    print("Pipeline Complete.")

In [37]:
await run_pipeline(df)

Processing Articles: 100%|██████████| 105542/105542 [59:29<00:00, 29.57it/s]

Pipeline Complete.


## Perform a search

In [ ]:
import httpx
from qdrant_client import AsyncQdrantClient

OLLAMA_URL = "http://ollama:11434/api/embeddings"
QDRANT_URL = "http://qdrant:6333"
COLLECTION_NAME = "fashion_articles"

async def search_fashion_articles(query_text, limit=5):
    async with httpx.AsyncClient() as http_client:
        payload = {
            "model": "embeddinggemma",
            "prompt": query_text
        }
        
        try:
            response = await http_client.post(OLLAMA_URL, json=payload, timeout=60.0)
            response.raise_for_status()
            query_vector = response.json()['embedding']

            async_qdrant = AsyncQdrantClient(url=QDRANT_URL)
            result = await async_qdrant.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector,
                limit=limit,
                with_payload=True
            )

            return result.points
            
        except Exception as e:
            print(f"Search failed: {e}")
            return []


In [8]:
results = await search_fashion_articles("red silk dress")
for hit in results:
    print(f"{hit.payload['prod_name']} - Score: {hit.score}")

PQ FEMI SILK MIX DRESS - Score: 0.5397787
CNY Siljan dress - Score: 0.5387325
SILJAN Dress - Score: 0.5285165
Nora dress - Score: 0.5271125
Maserati dress - Score: 0.5205582
